# Agent 5 — ESG Scoring

**What this notebook does:**  
Takes the raw ESG numbers from the provided datasets and converts them into clean, comparable scores (0–100) for E (Environmental), S (Social), G (Governance), and a combined ESG score.

**Why we don't just use one vendor ESG score:**  
Academic research (Berg, Koelbel & Rigobon 2022 — *Aggregate Confusion*) shows that ESG scores from different vendors disagree significantly. So we build our own transparent score from raw variables — this way we can explain every number to the investor panel.

**How to present this to investors:**  
> *Rather than relying on a black-box third-party ESG rating, we constructed our own transparent score from raw environmental, social, and governance variables. Each variable is normalised and weighted explicitly, so we can justify every number in front of the panel.*

In [1]:
import pandas as pd
import numpy as np
from datetime import date
import glob

# Load master dataset from notebook 01
master_files = sorted(glob.glob("../outputs/scores/master_dataset_*.csv"))
if not master_files:
    raise FileNotFoundError("Run notebook 01 first to create the master dataset.")

master = pd.read_csv(master_files[-1])
print(f"Master dataset loaded: {len(master)} companies")
print("Columns available:", list(master.columns))

Master dataset loaded: 57 companies
Columns available: ['companyName', 'ticker', 'isin', 'country', 'bics_sector', 'bics_industry', 'bics_sub_industry', 'bics_activity', 'market_cap_eur_bn', 'index_weight_pct', 'reporting_year', 'scope1_emissions_tco2e', 'scope2_emissions_market_tco2e', 'scope2_emissions_location_tco2e', 'scope3_emissions_tco2e', 'scope3_categories_reported', 'total_energy_consumption_gwh', 'renewable_energy_pct', 'energy_intensity_mwh_per_eur_m_revenue', 'water_withdrawal_m3', 'water_consumption_m3', 'water_stress_area_pct', 'waste_generated_tonnes', 'waste_recycled_pct', 'hazardous_waste_tonnes', 'employees_total', 'women_in_workforce_pct', 'gender_pay_gap_pct', 'employee_turnover_pct', 'training_hours_per_employee', 'work_related_injury_rate', 'fatalities', 'supplier_esg_audits_pct', 'community_investment_eur_m', 'carbon_intensity_tco2e_per_eur_m_revenue', 'sbti_status', 'net_zero_target_year', 'reporting_year_gov', 'board_size', 'board_independence_pct', 'women_on_

## Step 1 — Choose which variables to include

After running the cell above, look at the column list and decide which columns belong to E, S, and G.

**Examples of typical columns:**
- **E (Environmental):** GHG emissions, Scope 1, Scope 2, carbon intensity, water usage, energy consumption
- **S (Social):** employee turnover, workplace accidents, gender pay gap, employee training hours
- **G (Governance):** board independence, board gender diversity, CEO pay ratio, audit committee independence

In [2]:
E_VARS = [
    "scope1_emissions_tco2e",
    "carbon_intensity_tco2e_per_eur_m_revenue",
    "renewable_energy_pct",
    "water_withdrawal_m3",
    "waste_recycled_pct",
]

S_VARS = [
    "women_in_workforce_pct",
    "gender_pay_gap_pct",
    "employee_turnover_pct",
    "work_related_injury_rate",
    "training_hours_per_employee",
]

G_VARS = [
    "board_independence_pct",
    "women_on_board_pct",
    "audit_committee_independence_pct",
    "executive_pay_esg_linked_pct",
    "ceo_pay_ratio",
]

# True = lower is better (e.g. emissions), False = higher is better
INVERT = {
    "scope1_emissions_tco2e":                  True,
    "carbon_intensity_tco2e_per_eur_m_revenue": True,
    "renewable_energy_pct":                    False,
    "water_withdrawal_m3":                     True,
    "waste_recycled_pct":                      False,
    "women_in_workforce_pct":                  False,
    "gender_pay_gap_pct":                      True,
    "employee_turnover_pct":                   True,
    "work_related_injury_rate":                True,
    "training_hours_per_employee":             False,
    "board_independence_pct":                  False,
    "women_on_board_pct":                      False,
    "audit_committee_independence_pct":        False,
    "executive_pay_esg_linked_pct":            False,
    "ceo_pay_ratio":                           True,
}

ALL_VARS = E_VARS + S_VARS + G_VARS
print(f"E variables: {len(E_VARS)}")
print(f"S variables: {len(S_VARS)}")
print(f"G variables: {len(G_VARS)}")

E variables: 5
S variables: 5
G variables: 5


## Step 2 — Normalise all variables to 0–100

We use min-max normalisation: the best company in each variable scores 100, the worst scores 0, everyone else is in between. This makes all variables comparable regardless of their units.

In [3]:
def normalise(series, invert=False):
    """Scale a series to 0-100. If invert=True, lower values get higher scores."""
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(50.0, index=series.index)  # all same value → 50
    scaled = (series - mn) / (mx - mn) * 100
    return 100 - scaled if invert else scaled

scores = master[["ticker"] if "ticker" in master.columns else [master.columns[0]]].copy()

# Normalise each variable
for col in ALL_VARS:
    if col in master.columns:
        inv = INVERT.get(col, False)
        scores[f"{col}_score"] = normalise(master[col], invert=inv)
    else:
        print(f"WARNING: Column '{col}' not found in master dataset")

print("Variables normalised.")
scores.head(3)

Variables normalised.


,ticker,scope1_emissions_tco2e_score,carbon_intensity_tco2e_per_eur_m_revenue_score,renewable_energy_pct_score,water_withdrawal_m3_score,waste_recycled_pct_score,women_in_workforce_pct_score,gender_pay_gap_pct_score,employee_turnover_pct_score,work_related_injury_rate_score,training_hours_per_employee_score,board_independence_pct_score,women_on_board_pct_score,audit_committee_independence_pct_score,executive_pay_esg_linked_pct_score,ceo_pay_ratio_score
0,ASML.AS,94.033675,96.093311,22.150259,93.736489,77.394035,54.846939,52.842809,59.060403,83.333333,11.515152,16.222222,27.536232,31.174089,13.281250,87.570621
1,SAP.DE,94.270716,97.423487,68.005181,97.548217,89.010989,78.316327,65.886288,42.281879,65.981735,4.040404,78.666667,42.028986,74.898785,73.697917,32.768362
2,CAP.PA,98.512746,97.865659,62.564767,99.989476,34.379906,73.979592,65.217391,18.791946,75.570776,38.181818,57.555556,13.768116,25.910931,17.968750,92.090395


## Step 3 — Calculate E, S, G and combined ESG scores

We weight each pillar equally (33%/33%/33%) unless you want to change this.  
The combined ESG score is a weighted average of E, S, and G.

In [4]:
# Weights for E, S, G pillars (must sum to 1.0)
E_WEIGHT = 0.40   # 40% — heavier weight on environmental for sustainability focus
S_WEIGHT = 0.30   # 30%
G_WEIGHT = 0.30   # 30%

assert abs(E_WEIGHT + S_WEIGHT + G_WEIGHT - 1.0) < 0.001, "Weights must sum to 1.0"

def pillar_score(df, variables):
    cols = [f"{v}_score" for v in variables if f"{v}_score" in df.columns]
    if not cols:
        return pd.Series(np.nan, index=df.index)
    return df[cols].mean(axis=1)  # simple average across variables

scores["E_score"] = pillar_score(scores, E_VARS).round(2)
scores["S_score"] = pillar_score(scores, S_VARS).round(2)
scores["G_score"] = pillar_score(scores, G_VARS).round(2)

scores["ESG_score"] = (
    scores["E_score"].fillna(50) * E_WEIGHT +
    scores["S_score"].fillna(50) * S_WEIGHT +
    scores["G_score"].fillna(50) * G_WEIGHT
).round(2)

final_scores = scores[[c for c in scores.columns if not c.endswith("_score") or c in ["E_score","S_score","G_score","ESG_score"]]]

print("Top 10 by ESG score:")
final_scores.nlargest(10, "ESG_score")[["ticker","E_score","S_score","G_score","ESG_score"]]

Top 10 by ESG score:


,ticker,E_score,S_score,G_score,ESG_score
30,ALV.DE,84.34,63.67,67.24,73.01
25,BNP.PA,79.52,63.93,72.57,72.76
29,CS.PA,91.20,54.94,63.78,72.10
56,AMD,97.97,58.59,50.13,71.80
39,ITX.MC,89.85,62.83,56.39,71.71
54,VNA.DE,88.81,45.68,72.83,71.08
1,SAP.DE,89.25,51.30,60.41,69.21
40,ADS.DE,85.58,59.35,56.04,68.85
52,ORA.PA,73.69,69.20,58.29,67.72
41,RI.PA,73.53,57.39,65.92,66.40


## Step 4 — WACI (Weighted Average Carbon Intensity)

WACI is the standard carbon metric required by the assignment.  
It measures how carbon-intensive the portfolio is, weighted by how much we invest in each company.  
Unit: tonnes of CO₂ per million euros of revenue (tCO₂e / €m revenue)

In [5]:
# carbon_intensity_tco2e_per_eur_m_revenue already computed in the master dataset
CARBON_INTENSITY_COL = "carbon_intensity_tco2e_per_eur_m_revenue"

if CARBON_INTENSITY_COL in master.columns:
    ci = master[["ticker", CARBON_INTENSITY_COL]].copy().dropna()
    ci.columns = ["ticker", "carbon_intensity"]
    print("Carbon intensity (tCO2e per EUR m revenue) — top 10 lowest (cleanest):")
    print(ci.sort_values("carbon_intensity").head(10).to_string(index=False))
    print("\nTop 10 highest (most carbon-intensive):")
    print(ci.sort_values("carbon_intensity", ascending=False).head(10).to_string(index=False))
else:
    print("Column not found:", CARBON_INTENSITY_COL)

Carbon intensity (tCO2e per EUR m revenue) — top 10 lowest (cleanest):
  ticker  carbon_intensity
     AMD              1.98
  SAN.PA             16.30
  BNP.PA            134.70
  ROG.SW            145.50
   OR.PA            204.50
STLAM.MI            253.00
   CS.PA            308.50
  ORA.PA            354.00
 INGA.AS            413.50
  IBE.MC            459.00

Top 10 highest (most carbon-intensive):
   ticker  carbon_intensity
   BAS.DE           30035.6
   SHEL.L           29426.9
  ENEL.MI           27055.0
ORSTED.CO           25155.8
   TTE.PA           14990.8
  EQNR.OL           11707.0
   ULVR.L           10513.1
    RI.PA            9017.6
    MT.AS            8871.4
  HOLN.SW            7241.1


## Step 5 — Save ESG scores

In [6]:
today = str(date.today())
final_scores["data_vintage"] = today
out_path = f"../outputs/scores/esg_scores_{today}.csv"
final_scores.to_csv(out_path, index=False)
print(f"ESG scores saved: {out_path}")
print(f"Score summary:")
final_scores[["E_score","S_score","G_score","ESG_score"]].describe().round(2)

ESG scores saved: ../outputs/scores/esg_scores_2026-05-06.csv
Score summary:


C:\Users\ionva\AppData\Local\Temp\ipykernel_2200\3617176940.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_scores["data_vintage"] = today


,E_score,S_score,G_score,ESG_score
count,57.00,57.00,57.00,57.00
mean,70.97,52.20,51.53,59.51
std,15.78,11.75,10.96,8.05
min,22.93,22.19,26.63,34.31
25%,65.16,43.74,41.80,55.77
50%,73.53,52.35,50.99,59.66
75%,79.91,61.22,59.73,64.99
max,97.97,72.76,72.83,73.01


## ✅ Notebook complete

You now have:
- **E, S, G and combined ESG scores** (0–100) for all companies
- **Carbon intensity** per company
- A fully **transparent and explainable methodology** (not a black box)

**Next:** Open `04_portfolio_construction.ipynb` to select the final 15–25 holdings.